## 日本語ウェイクワードモデルの学習 (Google Colab対応版)


In [1]:
# @title 1. Google Cloud APIキーの設定 { display-mode: "form" }
# @markdown Google Cloud Text-to-Speech APIを使用して日本語音声を生成します。
# @markdown
# @markdown [こちら](https://cloud.google.com/text-to-speech/docs/quickstart)からAPIキーを取得してください。
# @markdown
# @markdown **重要**: APIキーは安全に管理してください。公開リポジトリにコミットしないよう注意！

import os

# APIキーを設定（フォーム入力）
api_key = "" # @param {type:"string"}

# 環境変数に設定
os.environ['GOOGLE_CLOUD_API_KEY'] = api_key

# 確認
if api_key:
    print("✅ APIキーが設定されました")
    print(f"   キーの長さ: {len(api_key)}文字")
else:
    print("❌ APIキーが設定されていません")
    print("   右側のフォームにAPIキーを入力してください")

# 作業ディレクトリの確認
print(f"\n📁 作業ディレクトリ: {os.getcwd()}")

❌ APIキーが設定されていません
   右側のフォームにAPIキーを入力してください

📁 作業ディレクトリ: /Users/miyatech/AndroidStudioProjects/openwakewordandroidkt


In [ ]:
# @title 2. データのダウンロード { display-mode: "form" }
# @markdown 高品質なモデルを学習するために必要なデータをダウンロードします。
# @markdown このプロセスには約15分かかります。

# @markdown **注意:** ダウンロードされるデータには様々なライセンスが含まれています。
# @markdown 個人的な**非商用**利用のみを想定してください。

import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

# openWakeWordをインストール
print("📥 openWakeWordをインストール中...")
!git clone https://github.com/dscripka/openwakeword 2>/dev/null || echo "Already cloned"
!pip install -e ./openwakeword -q

# 依存関係をインストール
print("\n📦 依存関係をインストール中...")
!pip install 'torch<=2.5' torchvision torchaudio -q
!pip install mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 -q
!pip install speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 -q
!pip install acoustics==0.2.6 onnx_tf==1.10.0 onnx2tf onnx ai_edge_litert==1.2.0 -q
!pip install onnx_graphsurgeon sng4onnx pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19 -q
!pip install jaconv -q  # 日本語 adversarial text生成用
!pip install google-cloud-texttospeech -q  # Google Cloud TTS API

# 必要なモデルをダウンロード
print("\n📥 基本モデルをダウンロード中...")
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

print("✅ 基本モデルをダウンロードしました")

# インポート
import numpy as np
import torch
from pathlib import Path
import yaml
import datasets
import scipy
from tqdm import tqdm

# MITルームインパルスレスポンスをダウンロード
output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    print("\n📥 ルームインパルスレスポンスをダウンロード中...")
    os.mkdir(output_dir)
    !git lfs install
    !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses 2>/dev/null
    rir_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("./MIT_environmental_impulse_responses/16khz").glob("*.wav")]}).cast_column("audio", datasets.Audio())
    for row in tqdm(rir_dataset, desc="RIRs保存中"):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# AudioSetデータをダウンロード
if not os.path.exists("audioset"):
    print("\n📥 背景音声データをダウンロード中...")
    os.mkdir("audioset")
    fname = "bal_train09.tar"
    out_dir = f"audioset/{fname}"
    link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
    !wget -q -O {out_dir} {link}
    !cd audioset && tar -xf bal_train09.tar

    output_dir = "./audioset_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset, desc="AudioSet変換中"):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# FMAデータセットをダウンロード
output_dir = "./fma"
if not os.path.exists(output_dir):
    print("\n📥 音楽データをダウンロード中...")
    os.mkdir(output_dir)
    fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
    fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

    n_hours = 1
    for i in tqdm(range(n_hours*3600//30), desc="FMA保存中"):
        row = next(fma_dataset)
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# 事前計算済みのopenWakeWord特徴量をダウンロード
if not os.path.exists("./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
    print("\n📥 学習用特徴量をダウンロード中（大きいファイルです）...")
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

if not os.path.exists("validation_set_features.npy"):
    print("\n📥 検証用特徴量をダウンロード中...")
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

print("\n✅ すべてのデータのダウンロードが完了しました！")

📥 openWakeWordをインストール中...
zsh:1: command not found: pip

📦 依存関係をインストール中...
zsh:1: command not found: pip
zsh:1: command not found: pip
zsh:1: command not found: pip
zsh:1: command not found: pip
zsh:1: command not found: pip
zsh:1: command not found: pip


UnboundLocalError: local variable 'child' referenced before assignment

--- Logging error ---
Traceback (most recent call last):
  File "/Users/miyatech/Library/Python/3.9/lib/python/site-packages/IPython/utils/_process_posix.py", line 148, in system
    child = pexpect.spawn(self.sh, args=['-c', cmd])  # Vanilla Pexpect
  File "/Users/miyatech/Library/Python/3.9/lib/python/site-packages/pexpect/pty_spawn.py", line 205, in __init__
    self._spawn(command, args, preexec_fn, dimensions)
  File "/Users/miyatech/Library/Python/3.9/lib/python/site-packages/pexpect/pty_spawn.py", line 303, in _spawn
    self.ptyproc = self._spawnpty(self.args, env=self.env,
  File "/Users/miyatech/Library/Python/3.9/lib/python/site-packages/pexpect/pty_spawn.py", line 315, in _spawnpty
    return ptyprocess.PtyProcess.spawn(args, **kwargs)
  File "/Users/miyatech/Library/Python/3.9/lib/python/site-packages/ptyprocess/ptyprocess.py", line 269, in spawn
    os.closerange(pair[0]+1, pair[1])
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Tr

In [ ]:
# @title 3. 日本語対応generate_samplesの準備 { display-mode: "form" }
# @markdown 日本語の学習データ生成用モジュールをセットアップします。

# 作業ディレクトリを設定
import os, sys
work_dir = os.getcwd()
print(f"作業ディレクトリ: {work_dir}")

# piper-sample-generatorのディレクトリを作成
os.makedirs("./piper-sample-generator", exist_ok=True)

# 日本語対応のgenerate_samples.pyを作成（エラーハンドリング強化版）
generate_samples_content = '''"""日本語対応 generate_samples.py for Google Colab (エラーハンドリング強化版)"""
import os
import sys
import random
import numpy as np
import scipy.io.wavfile
import time
import logging
import traceback
from typing import List, Optional, Union
from pathlib import Path

# tqdmのインポートを試みる
try:
    from tqdm import tqdm
except ImportError:
    # tqdmがない場合はダミー関数を使用
    def tqdm(iterable, desc="", total=None):
        return iterable

# google.cloudのインポートを遅延させる
texttospeech = None

def _import_texttospeech():
    global texttospeech
    if texttospeech is None:
        try:
            from google.cloud import texttospeech as tts
            texttospeech = tts
        except ImportError:
            raise ImportError("google-cloud-texttospeech is not installed")
    return texttospeech

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# 日本語Chirp3 HD音声リスト（高品質音声）
JAPANESE_CHIRP3_HD_VOICES = {
    "ja-JP-Chirp3-HD-Achernar": "FEMALE",
    "ja-JP-Chirp3-HD-Achird": "MALE",
    "ja-JP-Chirp3-HD-Algenib": "MALE",
    "ja-JP-Chirp3-HD-Algieba": "MALE",
    "ja-JP-Chirp3-HD-Alnilam": "MALE",
    "ja-JP-Chirp3-HD-Aoede": "FEMALE",
    "ja-JP-Chirp3-HD-Autonoe": "FEMALE",
    "ja-JP-Chirp3-HD-Callirrhoe": "FEMALE",
    "ja-JP-Chirp3-HD-Charon": "MALE",
    "ja-JP-Chirp3-HD-Despina": "FEMALE",
    "ja-JP-Chirp3-HD-Enceladus": "MALE",
    "ja-JP-Chirp3-HD-Erinome": "FEMALE",
    "ja-JP-Chirp3-HD-Fenrir": "MALE",
    "ja-JP-Chirp3-HD-Gacrux": "FEMALE",
    "ja-JP-Chirp3-HD-Iapetus": "MALE",
    "ja-JP-Chirp3-HD-Kore": "FEMALE",
    "ja-JP-Chirp3-HD-Laomedeia": "FEMALE",
    "ja-JP-Chirp3-HD-Leda": "FEMALE",
    "ja-JP-Chirp3-HD-Orus": "MALE",
    "ja-JP-Chirp3-HD-Puck": "MALE",
    "ja-JP-Chirp3-HD-Pulcherrima": "FEMALE",
    "ja-JP-Chirp3-HD-Rasalgethi": "MALE",
    "ja-JP-Chirp3-HD-Sadachbia": "MALE",
    "ja-JP-Chirp3-HD-Sadaltager": "MALE",
    "ja-JP-Chirp3-HD-Schedar": "MALE",
    "ja-JP-Chirp3-HD-Sulafat": "FEMALE",
    "ja-JP-Chirp3-HD-Umbriel": "MALE",
    "ja-JP-Chirp3-HD-Vindemiatrix": "FEMALE",
    "ja-JP-Chirp3-HD-Zephyr": "FEMALE",
    "ja-JP-Chirp3-HD-Zubenelgenubi": "MALE"
}

# 速度・ピッチ調整用の標準音声（Chirp3 HDは速度・ピッチ調整不可）
JAPANESE_STANDARD_VOICES = [
    "ja-JP-Neural2-B",  # Male
    "ja-JP-Neural2-C",  # Male
    "ja-JP-Neural2-D",  # Male
    "ja-JP-Wavenet-A",  # Female
    "ja-JP-Wavenet-B",  # Female
    "ja-JP-Wavenet-C",  # Male
    "ja-JP-Wavenet-D",  # Male
]

# オーディオデバイスプロファイル（追加のバリエーション用）
AUDIO_PROFILES = [
    "wearable-class-device",
    "handset-class-device",
    "headphone-class-device",
    "small-bluetooth-speaker-class-device",
    "medium-bluetooth-speaker-class-device",
    "large-home-entertainment-class-device",
]

class RateLimiter:
    def __init__(self, calls_per_minute=900):
        self.calls_per_minute = calls_per_minute
        self.interval = 60.0 / calls_per_minute
        self.last_call = 0

    def wait_if_needed(self):
        current = time.time()
        elapsed = current - self.last_call
        if elapsed < self.interval:
            time.sleep(self.interval - elapsed)
        self.last_call = time.time()

rate_limiter = RateLimiter(900)

def is_japanese(text):
    import re
    return bool(re.search(r'[ぁ-ゔァ-ヴー々〆〤一-龥]', text))

def filter_japanese_texts(text_list: List[str]) -> List[str]:
    """Filter out empty strings and non-Japanese texts from the list."""
    if not text_list:
        return []

    filtered_texts = []
    for text in text_list:
        if text and text.strip() and is_japanese(text.strip()):
            filtered_texts.append(text.strip())

    return filtered_texts

def generate_samples(
    text: Union[str, List[str]],
    max_samples: int = 100,
    length_scales: List[float] = [1.0],
    noise_scales: List[float] = [1.0],
    noise_scale_ws: List[float] = [1.0],
    output_dir: str = "./",
    batch_size: int = 10,
    auto_reduce_batch_size: bool = True,
    file_names: Optional[List[str]] = None
):
    """
    Generate Japanese TTS samples with enhanced error handling and debugging.
    """
    try:
        logger.info(f"=== generate_samples called ===")
        logger.info(f"Input text type: {type(text)}")
        logger.info(f"Input text (first 5): {text[:5] if isinstance(text, list) else text}")
        logger.info(f"Max samples: {max_samples}")
        logger.info(f"Output dir: {output_dir}")

        if isinstance(text, str):
            text = [text]

        # Create output directory
        os.makedirs(output_dir, exist_ok=True)
        logger.info(f"Created/verified output directory: {output_dir}")

        existing_files = list(Path(output_dir).glob("*.wav"))
        start_idx = len(existing_files)
        logger.info(f"Found {len(existing_files)} existing WAV files, starting from index {start_idx}")

        # Filter and validate Japanese texts with detailed logging
        original_text_count = len(text) if text else 0
        logger.info(f"Original text list length: {original_text_count}")

        if text:
            filtered_texts = filter_japanese_texts(text)
            logger.info(f"After filtering: {len(filtered_texts)} valid Japanese texts")
            if filtered_texts:
                text = filtered_texts
                logger.info(f"Using {len(text)} valid Japanese texts for generation")
                logger.info(f"Sample texts: {text[:3]}")
            else:
                logger.warning("No valid Japanese texts found in input, using default phrases")
                text = []
        else:
            logger.warning("Input text list was empty")

        # Handle empty or invalid text lists
        if not text:
            logger.warning("Empty or invalid text list provided, using default negative phrases")
            # ターゲットワードに基づいて自動的に類似語を生成
            text = ["トモニ", "コモリ", "トマリ", "ドモリ", "トボリ"]
            logger.info(f"Using fallback phrases: {text}")

        logger.info(f"Final text list for generation: {text}")
        logger.info(f"Generating {max_samples} Japanese samples...")

        # Check API key
        api_key = os.environ.get('GOOGLE_CLOUD_API_KEY', '')
        if not api_key:
            logger.error("GOOGLE_CLOUD_API_KEY not set!")
            raise RuntimeError("GOOGLE_CLOUD_API_KEY not set")
        else:
            logger.info(f"API key found (length: {len(api_key)})")

        # Google Cloud TTSをインポート
        logger.info("Importing Google Cloud TTS...")
        tts = _import_texttospeech()
        logger.info("TTS imported successfully")

        logger.info("Creating TTS client...")
        client = tts.TextToSpeechClient(
            client_options={"api_key": api_key}
        )
        logger.info("TTS client created successfully")

        # 使用する音声リスト（Chirp3 HD音声を優先使用）
        use_chirp3 = True
        mix_voices = True

        if use_chirp3 and not mix_voices:
            voices = list(JAPANESE_CHIRP3_HD_VOICES.keys())
        elif not use_chirp3:
            voices = JAPANESE_STANDARD_VOICES
        else:
            # 最大のバリエーションのために両方を混ぜる
            voices = list(JAPANESE_CHIRP3_HD_VOICES.keys()) + JAPANESE_STANDARD_VOICES

        logger.info(f"Using {len(voices)} voices for generation")

        # プログレスバーの設定
        pbar = tqdm(total=max_samples, desc="Generating Japanese samples")

        generated_count = 0
        consecutive_errors = 0
        max_consecutive_errors = 10

        while generated_count < max_samples and consecutive_errors < max_consecutive_errors:
            try:
                current_text = text[generated_count % len(text)]
                voice_name = random.choice(voices)

                # パラメータのランダム化（連続的な値を使用）
                # Speed: 0.5〜1.75の範囲で連続的にランダム
                speaking_rate = random.uniform(0.5, 1.75)
                
                # Pitch: -5〜+5セミトーンの範囲で連続的にランダム
                pitch = random.uniform(-5.0, 5.0)

                rate_limiter.wait_if_needed()

                synthesis_input = tts.SynthesisInput(text=current_text)
                voice = tts.VoiceSelectionParams(
                    language_code="ja-JP",
                    name=voice_name
                )
                audio_config = tts.AudioConfig(
                    audio_encoding=tts.AudioEncoding.LINEAR16,
                    sample_rate_hertz=16000
                )

                # Chirp3 HD以外の音声のみ速度とピッチを調整
                if "Chirp3" not in voice_name:
                    audio_config.speaking_rate = speaking_rate
                    audio_config.pitch = pitch

                # オーディオプロファイルをランダムに適用（50%の確率）
                if random.random() < 0.5:
                    audio_profile = random.choice(AUDIO_PROFILES)
                    audio_config.effects_profile_id = [audio_profile]

                # TTS API call with detailed error logging
                try:
                    response = client.synthesize_speech(
                        input=synthesis_input,
                        voice=voice,
                        audio_config=audio_config
                    )
                except Exception as tts_error:
                    logger.error(f"TTS API error for text '{current_text}' with voice '{voice_name}': {tts_error}")
                    consecutive_errors += 1
                    if "429" in str(tts_error) or "quota" in str(tts_error).lower():
                        logger.warning("Rate limit or quota hit, waiting 60 seconds...")
                        time.sleep(60)
                        consecutive_errors = 0  # Reset error count after wait
                    elif "Chirp3" in voice_name:
                        logger.info("Chirp3 error, falling back to standard voices...")
                        voices = JAPANESE_STANDARD_VOICES
                        consecutive_errors = 0  # Reset error count after fallback
                    continue

                # Determine output filename
                if file_names and generated_count < len(file_names):
                    output_file = os.path.join(output_dir, file_names[generated_count])
                else:
                    output_file = os.path.join(output_dir, f"{start_idx + generated_count:06d}.wav")

                # Save audio file with error handling
                try:
                    audio_array = np.frombuffer(response.audio_content, dtype=np.int16)
                    scipy.io.wavfile.write(output_file, 16000, audio_array)
                    logger.debug(f"Saved audio file: {output_file}")
                except Exception as save_error:
                    logger.error(f"Error saving audio file {output_file}: {save_error}")
                    consecutive_errors += 1
                    continue

                generated_count += 1
                consecutive_errors = 0  # Reset consecutive error count on success
                pbar.update(1)

                # Log progress every 50 samples
                if generated_count % 50 == 0:
                    logger.info(f"Generated {generated_count}/{max_samples} samples")

            except Exception as e:
                logger.error(f"Unexpected error in generation loop: {e}")
                logger.error(f"Traceback: {traceback.format_exc()}")
                consecutive_errors += 1
                if consecutive_errors >= max_consecutive_errors:
                    logger.error(f"Too many consecutive errors ({consecutive_errors}), stopping generation")
                    break
                continue

        pbar.close()

        # Final summary
        final_file_count = len(list(Path(output_dir).glob("*.wav")))
        logger.info(f"=== Generation completed ===")
        logger.info(f"Generated {generated_count}/{max_samples} samples successfully")
        logger.info(f"Total WAV files in output directory: {final_file_count}")
        logger.info(f"Directory: {output_dir}")

        if generated_count == 0:
            logger.error("No samples were generated! Check the logs above for errors.")
            # List directory contents for debugging
            try:
                dir_contents = os.listdir(output_dir)
                logger.info(f"Directory contents: {dir_contents}")
            except Exception as e:
                logger.error(f"Could not list directory contents: {e}")

        return generated_count

    except Exception as e:
        logger.error(f"Fatal error in generate_samples: {e}")
        logger.error(f"Traceback: {traceback.format_exc()}")
        raise
'''

with open("./piper-sample-generator/generate_samples.py", "w", encoding="utf-8") as f:
    f.write(generate_samples_content)

# __init__.pyも作成（空でOK）
with open("./piper-sample-generator/__init__.py", "w") as f:
    f.write("")

print("✅ 日本語対応generate_samplesを作成しました（詳細デバッグ版）")

# sys.pathに追加
if "piper-sample-generator/" not in sys.path:
    sys.path.insert(0, os.path.join(work_dir, "piper-sample-generator"))

In [4]:
# @title 4. 日本語 Adversarial Text 生成モジュールの準備 { display-mode: "form" }
# @markdown 日本語のロバスト性向上のための類似語生成モジュールを作成します。

import os
import sys

# 現在のディレクトリを確認
print(f"現在のディレクトリ: {os.getcwd()}")

# openwakeword/openwakeword ディレクトリが存在することを確認
openwakeword_dir = "./openwakeword"
openwakeword_subdir = "./openwakeword/openwakeword"

print(f"openwakewordディレクトリ確認:")
print(f"  {openwakeword_dir} 存在: {os.path.exists(openwakeword_dir)}")
print(f"  {openwakeword_subdir} 存在: {os.path.exists(openwakeword_subdir)}")

if not os.path.exists(openwakeword_dir):
    print("❌ openwakewordディレクトリが見つかりません。cell 2を先に実行してください。")
elif not os.path.exists(openwakeword_subdir):
    print("❌ openwakeword/openwakewordディレクトリが見つかりません。")
    print(f"   ディレクトリ内容: {os.listdir(openwakeword_dir)}")
else:
    print("✅ ディレクトリ構造OK")

# data_ja.pyを作成（改良版 - 堅牢性とエラーハンドリング強化）
data_ja_content = '''"""
Japanese adversarial text generation for openWakeWord
Generates phonetically similar Japanese text for robustness training
"""

import itertools
import random
import numpy as np
import logging
from typing import List, Tuple, Dict, Optional
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing

# Try importing Japanese text processing libraries
try:
    import jaconv
except ImportError:
    jaconv = None
    logging.warning("jaconv not installed. Install with: pip install jaconv")

# Similar kana mapping based on phonetic similarity
SIMILAR_KANA_MAP = {
    # Hiragana mappings
    'あ': ['お', 'わ'], 'い': ['え'], 'う': ['お'], 'え': ['い'], 'お': ['あ', 'う', 'を'],

    # K-group (ka-row) clear/voiced
    'か': ['が', 'き'], 'が': ['か'], 'き': ['ぎ', 'く'], 'ぎ': ['き'],
    'く': ['ぐ', 'き'], 'ぐ': ['く'], 'け': ['げ', 'く'], 'げ': ['け'],
    'こ': ['ご', 'く'], 'ご': ['こ'],

    # S-group clear/voiced
    'さ': ['ざ', 'し'], 'ざ': ['さ'], 'し': ['じ', 'ち'], 'じ': ['し', 'ぢ'],
    'す': ['ず', 'つ'], 'ず': ['す', 'づ'], 'せ': ['ぜ', 'て'], 'ぜ': ['せ'],
    'そ': ['ぞ', 'と'], 'ぞ': ['そ'],

    # T-group clear/voiced
    'た': ['だ', 'ち'], 'だ': ['た'], 'ち': ['ぢ', 'し'], 'ぢ': ['じ', 'ち'],
    'つ': ['づ', 'す'], 'づ': ['ず', 'つ'], 'て': ['で', 'せ'], 'で': ['て'],
    'と': ['ど', 'そ'], 'ど': ['と'],

    # N-group and R-group similarity
    'な': ['ら', 'ま'], 'に': ['り', 'み'], 'ぬ': ['る', 'む'], 'ね': ['れ', 'め'], 'の': ['ろ', 'も'],
    'ら': ['な', 'だ'], 'り': ['に', 'い'], 'る': ['ぬ', 'う'], 'れ': ['ね', 'え'], 'ろ': ['の', 'お'],

    # H-group clear/voiced/p-sound
    'は': ['ば', 'ぱ', 'わ'], 'ば': ['は', 'ぱ'], 'ぱ': ['は', 'ば'],
    'ひ': ['び', 'ぴ', 'い'], 'び': ['ひ', 'ぴ'], 'ぴ': ['ひ', 'び'],
    'ふ': ['ぶ', 'ぷ', 'う'], 'ぶ': ['ふ', 'ぷ'], 'ぷ': ['ふ', 'ぶ'],
    'へ': ['べ', 'ぺ', 'え'], 'べ': ['へ', 'ぺ'], 'ぺ': ['へ', 'べ'],
    'ほ': ['ぼ', 'ぽ', 'お'], 'ぼ': ['ほ', 'ぽ'], 'ぽ': ['ほ', 'ぼ'],

    # M-group
    'ま': ['な', 'も'], 'み': ['に', 'め'], 'む': ['ぬ', 'も'], 'め': ['ね', 'み'], 'も': ['の', 'ま'],

    # Y-group and W-group
    'や': ['わ'], 'ゆ': ['よ'], 'よ': ['ゆ'], 'わ': ['は', 'あ', 'や'], 'を': ['お'], 'ん': ['む'],

    # Small kana
    'ゃ': ['ゅ', 'ょ'], 'ゅ': ['ゃ', 'ょ'], 'ょ': ['ゃ', 'ゅ'],
    'っ': ['つ'], 'ー': ['い', 'う'],

    # Katakana mappings (same pattern as hiragana)
    'ア': ['オ', 'ワ'], 'イ': ['エ'], 'ウ': ['オ'], 'エ': ['イ'], 'オ': ['ア', 'ウ', 'ヲ'],

    'カ': ['ガ', 'キ'], 'ガ': ['カ'], 'キ': ['ギ', 'ク'], 'ギ': ['キ'],
    'ク': ['グ', 'キ'], 'グ': ['ク'], 'ケ': ['ゲ', 'ク'], 'ゲ': ['ケ'],
    'コ': ['ゴ', 'ク'], 'ゴ': ['コ'],

    'サ': ['ザ', 'シ'], 'ザ': ['サ'], 'シ': ['ジ', 'チ'], 'ジ': ['シ', 'ヂ'],
    'ス': ['ズ', 'ツ'], 'ズ': ['ス', 'ヅ'], 'セ': ['ゼ', 'テ'], 'ゼ': ['セ'],
    'ソ': ['ゾ', 'ト'], 'ゾ': ['ソ'],

    'タ': ['ダ', 'チ'], 'ダ': ['タ'], 'チ': ['ヂ', 'シ'], 'ヂ': ['ジ', 'チ'],
    'ツ': ['ヅ', 'ス'], 'ヅ': ['ズ', 'ツ'], 'テ': ['デ', 'セ'], 'デ': ['テ'],
    'ト': ['ド', 'ソ'], 'ド': ['ト'],

    'ナ': ['ラ', 'マ'], 'ニ': ['リ', 'ミ'], 'ヌ': ['ル', 'ム'], 'ネ': ['レ', 'メ'], 'ノ': ['ロ', 'モ'],
    'ラ': ['ナ', 'ダ'], 'リ': ['ニ', 'イ'], 'ル': ['ヌ', 'ウ'], 'レ': ['ネ', 'エ'], 'ロ': ['ノ', 'オ'],

    'ハ': ['バ', 'パ', 'ワ'], 'バ': ['ハ', 'パ'], 'パ': ['ハ', 'バ'],
    'ヒ': ['ビ', 'ピ', 'イ'], 'ビ': ['ヒ', 'ピ'], 'ピ': ['ヒ', 'ビ'],
    'フ': ['ブ', 'プ', 'ウ'], 'ブ': ['フ', 'プ'], 'プ': ['フ', 'ブ'],
    'ヘ': ['ベ', 'ペ', 'エ'], 'ベ': ['ヘ', 'ペ'], 'ペ': ['ヘ', 'ベ'],
    'ホ': ['ボ', 'ポ', 'オ'], 'ボ': ['ホ', 'ポ'], 'ポ': ['ホ', 'ボ'],

    'マ': ['ナ', 'モ'], 'ミ': ['ニ', 'メ'], 'ム': ['ヌ', 'モ'], 'メ': ['ネ', 'ミ'], 'モ': ['ノ', 'マ'],

    'ヤ': ['ワ'], 'ユ': ['ヨ'], 'ヨ': ['ユ'], 'ワ': ['ハ', 'ア', 'ヤ'], 'ヲ': ['オ'], 'ン': ['ム'],

    'ャ': ['ュ', 'ョ'], 'ュ': ['ャ', 'ョ'], 'ョ': ['ャ', 'ュ'],
    'ッ': ['ツ'], 'ー': ['イ', 'ウ'],
}


def kana_replacement(input_chars: List[str], max_replace: int) -> List[List[Optional[str]]]:
    """Generate replacement patterns for Japanese kana characters."""
    results = []
    chars = list(input_chars)

    # Generate all combinations for replacement (1 to max_replace)
    for r in range(1, min(max_replace + 1, len(chars) + 1)):
        # Get all combinations of positions to replace
        comb = itertools.combinations(range(len(chars)), r)
        for indices in comb:
            chars_copy = chars.copy()
            for i in indices:
                chars_copy[i] = None  # Mark for replacement
            results.append(chars_copy)

    return results


def generate_from_pattern_with_original(pattern: List[Optional[str]],
                                       original_chars: List[str],
                                       similar_map: Dict[str, List[str]]) -> List[str]:
    """Generate words from a pattern with original characters reference."""
    # Find replacement positions
    replace_positions = [i for i, char in enumerate(pattern) if char is None]

    if not replace_positions:
        return [''.join(pattern)]

    # Get all possible replacements for each position
    replacement_options = []
    for pos in replace_positions:
        original_char = original_chars[pos]
        similar_chars = similar_map.get(original_char, [])
        if similar_chars:
            replacement_options.append((pos, similar_chars))

    if not replacement_options:
        return []

    # Generate all combinations
    generated_words = []

    # Use itertools.product to get all combinations
    if len(replacement_options) == 1:
        pos, chars = replacement_options[0]
        for char in chars:
            word_chars = original_chars.copy()
            word_chars[pos] = char
            generated_words.append(''.join(word_chars))
    else:
        # For multiple replacements, sample to avoid explosion
        max_combinations = 50
        for _ in range(max_combinations):
            word_chars = original_chars.copy()
            for pos, chars in replacement_options:
                if chars:
                    word_chars[pos] = random.choice(chars)
            generated_words.append(''.join(word_chars))

    # Remove duplicates
    return list(set(generated_words))


def generate_simple_japanese_variants(text: str, N: int) -> List[str]:
    """
    Generate simple Japanese text variants without jaconv.
    This is a fallback function when jaconv is not available.
    """
    variants = []
    chars = list(text)

    # Generate variants for each character position
    for _ in range(N):
        if not chars:
            continue

        # Randomly select positions to modify
        n_changes = min(len(chars), max(1, len(chars) // 2))
        positions = random.sample(range(len(chars)), n_changes)

        variant_chars = chars.copy()
        for pos in positions:
            char = chars[pos]
            if char in SIMILAR_KANA_MAP:
                similar_chars = SIMILAR_KANA_MAP[char]
                if similar_chars:
                    variant_chars[pos] = random.choice(similar_chars)

        variant = ''.join(variant_chars)
        if variant != text:
            variants.append(variant)

    # Remove duplicates
    return list(set(variants))


def generate_japanese_adversarial_texts(input_text: str, N: int,
                                      include_partial_phrase: float = 0,
                                      include_input_words: float = 0) -> List[str]:
    """Generate adversarial Japanese texts based on kana similarity."""
    # Input validation
    if not input_text or not input_text.strip():
        logging.warning("Empty input text provided to generate_japanese_adversarial_texts")
        return []

    if N <= 0:
        logging.warning("Invalid N value provided to generate_japanese_adversarial_texts")
        return []

    # Validate that input is actually Japanese
    if not is_japanese(input_text):
        logging.warning(f"Input text '{input_text}' does not contain Japanese characters")
        return []

    # Ensure we have jaconv for conversion
    if jaconv is None:
        logging.warning("jaconv not available, using simple variant generation")
        # Fallback to simple version
        return generate_simple_japanese_variants(input_text, N)

    try:
        # Split input into words
        words = input_text.split()
        if not words:
            logging.warning("No words found after splitting input text")
            return []

        # Process each word to generate adversarial variants
        adversarial_phrases = []

        for word in words:
            if not word.strip():
                continue

            # Convert to list of characters
            chars = list(word.strip())

            adversarial_words = []

            try:
                # Short words: try all characters
                if len(chars) <= 2:
                    patterns = kana_replacement(chars, len(chars))
                else:
                    # Longer words: replace up to len-2 characters (keep some similarity)
                    patterns = kana_replacement(chars, max(1, len(chars) - 2))

                # Generate adversarial words from patterns
                for pattern in patterns:
                    try:
                        generated = generate_from_pattern_with_original(pattern, chars, SIMILAR_KANA_MAP)
                        # Keep only words different from original and ensure they are Japanese
                        valid_generated = [w for w in generated if w != word and w.strip() and is_japanese(w)]
                        adversarial_words.extend(valid_generated)
                    except Exception as e:
                        logging.debug(f"Error generating from pattern: {e}")
                        continue

                # Remove duplicates and limit number
                adversarial_words = list(set(adversarial_words))[:100]

                if adversarial_words:
                    adversarial_phrases.append(adversarial_words)
                else:
                    # If no adversarial words generated, add empty list
                    adversarial_phrases.append([])

            except Exception as e:
                logging.debug(f"Error processing word '{word}': {e}")
                adversarial_phrases.append([])

        # Generate final combinations (same logic as original)
        adversarial_texts = []

        for i in range(N):
            txts = []

            try:
                for j, (adv_list, orig_word) in enumerate(zip(adversarial_phrases, words)):
                    # Decide whether to keep original word
                    if adv_list and np.random.random() > include_input_words:
                        # Use adversarial word
                        txts.append(np.random.choice(adv_list))
                    else:
                        # Keep original word
                        txts.append(orig_word)

                # Handle partial phrases
                if include_partial_phrase > 0 and len(words) > 1 and np.random.random() <= include_partial_phrase:
                    # Generate partial phrase
                    n_words = np.random.randint(1, len(words) + 1)
                    selected_indices = np.random.choice(len(txts), size=n_words, replace=False)
                    selected_words = [txts[i] for i in sorted(selected_indices)]
                    generated_phrase = ' '.join(selected_words)
                else:
                    # Use full phrase
                    generated_phrase = ' '.join(txts)

                # Only add if it's different from input and is valid Japanese
                if generated_phrase != input_text and generated_phrase.strip() and is_japanese(generated_phrase):
                    adversarial_texts.append(generated_phrase)

            except Exception as e:
                logging.debug(f"Error in combination generation loop {i}: {e}")
                continue

        # Remove duplicates while preserving order
        seen = set()
        unique_texts = []
        for text in adversarial_texts:
            if text not in seen:
                seen.add(text)
                unique_texts.append(text)

        # Log results for debugging
        logging.info(f"Generated {len(unique_texts)} unique adversarial texts from '{input_text}'")

        return unique_texts

    except Exception as e:
        logging.error(f"Unexpected error in generate_japanese_adversarial_texts: {e}")
        # Fallback to simple generation
        return generate_simple_japanese_variants(input_text, N)


def generate_japanese_adversarial_texts_parallel(input_text: str, N: int,
                                               include_partial_phrase: float = 0,
                                               include_input_words: float = 0,
                                               n_workers: Optional[int] = None) -> List[str]:
    """Parallel version of generate_japanese_adversarial_texts for efficiency."""
    # If jaconv is not available, fall back to non-parallel simple version
    if jaconv is None:
        logging.warning("jaconv not available, using simple variant generation (non-parallel)")
        return generate_simple_japanese_variants(input_text, N)

    if n_workers is None:
        n_workers = min(multiprocessing.cpu_count(), 8)

    # Split work among workers
    samples_per_worker = N // n_workers
    remainder = N % n_workers

    futures = []
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        for i in range(n_workers):
            # Distribute remainder among first workers
            worker_samples = samples_per_worker + (1 if i < remainder else 0)

            if worker_samples > 0:
                future = executor.submit(
                    generate_japanese_adversarial_texts,
                    input_text, worker_samples,
                    include_partial_phrase, include_input_words
                )
                futures.append(future)

        # Collect results
        all_texts = []
        for future in as_completed(futures):
            try:
                texts = future.result()
                all_texts.extend(texts)
            except Exception as e:
                logging.error(f"Error in parallel generation: {e}")

    # Remove duplicates
    unique_texts = list(set(all_texts))

    # If we have more than needed, sample
    if len(unique_texts) > N:
        return random.sample(unique_texts, N)

    return unique_texts


# Helper function to check if text is Japanese
def is_japanese(text: str) -> bool:
    """Check if text contains Japanese characters."""
    import re
    return bool(re.search(r'[ぁ-ゔァ-ヴー々〆〤一-龥]', text))
'''

# data_ja.pyファイルのパス
data_ja_path = os.path.join(openwakeword_subdir, "data_ja.py")
print(f"\ndata_ja.py作成パス: {data_ja_path}")

try:
    # ファイルを作成
    with open(data_ja_path, "w", encoding="utf-8") as f:
        f.write(data_ja_content)

    print("✅ data_ja.py ファイルを作成しました")

    # ファイルが実際に作成されたかを確認
    if os.path.exists(data_ja_path):
        file_size = os.path.getsize(data_ja_path)
        print(f"✅ 作成確認: ファイルサイズ {file_size} bytes")

        # ファイルの先頭を読み込んで確認
        with open(data_ja_path, "r", encoding="utf-8") as f:
            first_lines = f.readlines()[:3]
        print(f"✅ ファイル内容確認: 先頭3行を読み込み成功")
        for i, line in enumerate(first_lines):
            print(f"  {i+1}: {line.rstrip()}")

    else:
        print("❌ ファイル作成に失敗しました")

except Exception as e:
    print(f"❌ ファイル作成エラー: {e}")
    import traceback
    print(f"詳細: {traceback.format_exc()}")

# sys.pathにopenwakewordディレクトリを追加
openwakeword_path = os.path.abspath(openwakeword_dir)
if openwakeword_path not in sys.path:
    sys.path.insert(0, openwakeword_path)
    print(f"\n✅ sys.pathに追加しました: {openwakeword_path}")

# インポートテスト
print(f"\n🔍 インポートテスト:")
try:
    from openwakeword.data_ja import is_japanese, generate_japanese_adversarial_texts
    print("✅ openwakeword.data_ja のインポート成功")

    # 簡単な機能テスト
    test_result = is_japanese("トモリ")
    print(f"✅ 機能テスト: is_japanese('トモリ') = {test_result}")

except ImportError as e:
    print(f"❌ インポートエラー: {e}")
    print(f"   sys.path: {sys.path[:2]}...")
    print(f"   openwakewordディレクトリ内容: {os.listdir(openwakeword_subdir) if os.path.exists(openwakeword_subdir) else 'ディレクトリなし'}")
except Exception as e:
    print(f"❌ その他のエラー: {e}")

print("\n📋 ファイル作成完了 - 次のセルで train.py を自動修正します")

現在のディレクトリ: /content
openwakewordディレクトリ確認:
  ./openwakeword 存在: True
  ./openwakeword/openwakeword 存在: True
✅ ディレクトリ構造OK

data_ja.py作成パス: ./openwakeword/openwakeword/data_ja.py
✅ data_ja.py ファイルを作成しました
✅ 作成確認: ファイルサイズ 15211 bytes
✅ ファイル内容確認: 先頭3行を読み込み成功
  1: """
  2: Japanese adversarial text generation for openWakeWord
  3: Generates phonetically similar Japanese text for robustness training

✅ sys.pathに追加しました: /content/openwakeword

🔍 インポートテスト:
✅ openwakeword.data_ja のインポート成功
✅ 機能テスト: is_japanese('トモリ') = True

📋 ファイル作成完了 - 次のセルで train.py を自動修正します


In [5]:
# @title 5. train.pyの自動修正とデバッグテスト { display-mode: "form" }
# @markdown 日本語のadversarial text生成をサポートするため、train.pyを修正します。
# @markdown また、generate_samples関数のテストも実行します。

import os
import sys
from pathlib import Path

# 詳細な診断情報を表示
print("🔍 システム診断:")
print(f"  現在のディレクトリ: {os.getcwd()}")
print(f"  Python実行パス: {sys.executable}")
print(f"  sys.path (最初の5項目):")
for i, path in enumerate(sys.path[:5]):
    print(f"    {i}: {path}")

# ディレクトリ構造の確認
print(f"\n📁 ディレクトリ構造確認:")
openwakeword_dir = "./openwakeword"
openwakeword_subdir = "./openwakeword/openwakeword"
data_ja_path = "./openwakeword/openwakeword/data_ja.py"

print(f"  {openwakeword_dir} 存在: {os.path.exists(openwakeword_dir)}")
print(f"  {openwakeword_subdir} 存在: {os.path.exists(openwakeword_subdir)}")
print(f"  {data_ja_path} 存在: {os.path.exists(data_ja_path)}")

if os.path.exists(data_ja_path):
    file_size = os.path.getsize(data_ja_path)
    print(f"  data_ja.py サイズ: {file_size} bytes")
else:
    print("  ❌ data_ja.py ファイルが見つかりません")

# sys.pathを設定
openwakeword_abs_path = os.path.abspath(openwakeword_dir)
if openwakeword_abs_path not in sys.path:
    sys.path.insert(0, openwakeword_abs_path)
    print(f"\n✅ sys.pathに追加: {openwakeword_abs_path}")

# __init__.py ファイルの確認と作成
init_py_path = os.path.join(openwakeword_subdir, "__init__.py")
if not os.path.exists(init_py_path):
    print(f"\n🔧 __init__.py を作成中: {init_py_path}")
    try:
        with open(init_py_path, "w") as f:
            f.write("# openWakeWord package initialization\n")
        print("✅ __init__.py を作成しました")
    except Exception as e:
        print(f"❌ __init__.py 作成エラー: {e}")

# Python キャッシュをクリア
if 'openwakeword.data_ja' in sys.modules:
    print("🔄 既存のモジュールキャッシュをクリア")
    del sys.modules['openwakeword.data_ja']

# インポートテスト（詳細エラー情報付き）
print(f"\n🔍 data_ja モジュールインポートテスト:")
try:
    # 段階的インポートテスト
    print("  Step 1: openwakeword パッケージをインポート")
    import openwakeword
    print(f"  ✅ openwakeword package path: {openwakeword.__file__}")

    print("  Step 2: openwakeword.data_ja をインポート")
    from openwakeword.data_ja import is_japanese, generate_japanese_adversarial_texts
    print("  ✅ openwakeword.data_ja のインポート成功")

    # 簡単な機能テスト
    print("  Step 3: 機能テスト")
    test_result = is_japanese("トモリ")
    print(f"  ✅ is_japanese('トモリ') = {test_result}")

    # adversarial texts生成テスト
    print("  Step 4: adversarial texts生成テスト")
    try:
        adversarial_texts = generate_japanese_adversarial_texts(
            "トモリ",
            N=5,
            include_partial_phrase=0.0,
            include_input_words=0.2
        )
        print(f"  ✅ 生成されたadversarial texts: {len(adversarial_texts)}")
        if adversarial_texts:
            print(f"  例: {adversarial_texts[:3]}")
        else:
            print("  ⚠️ adversarial texts は生成されませんでしたが、エラーなし")
    except Exception as adv_error:
        print(f"  ❌ adversarial texts生成エラー: {adv_error}")
        import traceback
        print(f"  詳細: {traceback.format_exc()}")

    JAPANESE_MODULE_AVAILABLE = True
    print("  ✅ 日本語モジュール利用可能")

except ImportError as e:
    print(f"  ❌ インポートエラー: {e}")
    print(f"  詳細調査:")

    # 詳細なディレクトリ内容確認
    if os.path.exists(openwakeword_subdir):
        print(f"    openwakeword/openwakeword/ 内容:")
        for item in os.listdir(openwakeword_subdir):
            item_path = os.path.join(openwakeword_subdir, item)
            if os.path.isfile(item_path):
                size = os.path.getsize(item_path)
                print(f"      📄 {item} ({size} bytes)")
            else:
                print(f"      📁 {item}/")

    # Python path情報
    print(f"    現在のPython path:")
    for i, path in enumerate(sys.path):
        if 'openwakeword' in path or i < 3:
            print(f"      {i}: {path}")

    JAPANESE_MODULE_AVAILABLE = False

except Exception as e:
    print(f"  ❌ その他のエラー: {e}")
    import traceback
    print(f"  詳細: {traceback.format_exc()}")
    JAPANESE_MODULE_AVAILABLE = False

# train.pyを修正して日本語adversarial text生成を有効化
print(f"\n🔧 train.py を修正中...")

train_py_path = "./openwakeword/openwakeword/train.py"

if not os.path.exists(train_py_path):
    print(f"❌ train.py が見つかりません: {train_py_path}")
else:
    try:
        # train.pyを読み込み
        with open(train_py_path, 'r', encoding='utf-8') as f:
            train_content = f.read()

        # 日本語サポートが既に追加されているか確認
        if "from openwakeword.data_ja import" not in train_content:
            print("  日本語サポートを追加中...")

            # インポート文を追加
            import_section = """from openwakeword.data import (
    load_wakeword_data, create_train_dataloader,
    create_validation_dataloader, get_negative_data,
    create_wakeword_dataset, generate_adversarial_texts
)"""

            new_import_section = """from openwakeword.data import (
    load_wakeword_data, create_train_dataloader,
    create_validation_dataloader, get_negative_data,
    create_wakeword_dataset, generate_adversarial_texts
)

# Import Japanese adversarial text generation
try:
    from openwakeword.data_ja import (
        generate_japanese_adversarial_texts,
        generate_japanese_adversarial_texts_parallel,
        is_japanese
    )
    JAPANESE_SUPPORT = True
except ImportError:
    JAPANESE_SUPPORT = False
    print("Warning: Japanese adversarial text generation not available")"""

            train_content = train_content.replace(import_section, new_import_section)

            # adversarial text生成部分を修正
            # Generate adversarial texts from the target phrase
            target_section = "                adversarial_texts = generate_adversarial_texts("

            # 修正後のコード
            new_section = """                # Check if Japanese text and use appropriate generator
                if JAPANESE_SUPPORT and config['target_phrase'] and is_japanese(config['target_phrase'][0]):
                    print(f"Generating Japanese adversarial texts for: {config['target_phrase'][0]}")
                    adversarial_texts = generate_japanese_adversarial_texts_parallel(
                        config['target_phrase'][0],
                        N=5000,
                        include_partial_phrase=1.0,
                        include_input_words=0.2
                    )
                    print(f"Generated {len(adversarial_texts)} Japanese adversarial texts")
                    if not adversarial_texts:
                        print("Warning: No Japanese adversarial texts generated, using fallback")
                        adversarial_texts = []
                else:
                    adversarial_texts = generate_adversarial_texts("""

            train_content = train_content.replace(target_section, new_section)

            # ファイルを保存
            with open(train_py_path, 'w', encoding='utf-8') as f:
                f.write(train_content)

            print("  ✅ train.py を修正しました - 日本語adversarial text生成が有効になりました")
        else:
            print("  ✅ train.py は既に日本語対応済みです")

    except Exception as e:
        print(f"  ❌ train.py修正エラー: {e}")
        import traceback
        print(f"  詳細: {traceback.format_exc()}")

# generate_samples 関数のテスト
print(f"\n🔍 generate_samples 関数のテストを実行中...")

# piper-sample-generator pathを設定
piper_path = os.path.join(os.getcwd(), "piper-sample-generator")
if piper_path not in sys.path:
    sys.path.insert(0, piper_path)

print(f"  piper-sample-generator path: {piper_path}")
print(f"  存在確認: {os.path.exists(piper_path)}")

try:
    # generate_samples をインポートしてテスト
    from generate_samples import generate_samples, is_japanese, filter_japanese_texts
    print("  ✅ generate_samples モジュールのインポート成功")

    # 1. 日本語判定テスト
    print(f"\n  📋 日本語判定テスト:")
    test_phrases = ["トモリ", "hello", "トモニ", "コモリ", "test123"]
    for phrase in test_phrases:
        result = is_japanese(phrase)
        print(f"    '{phrase}' → {result}")

    # 2. フィルタリングテスト
    print(f"\n  📋 フィルタリングテスト:")
    mixed_texts = ["トモリ", "hello", "トモニ", "", " ", "コモリ", "test"]
    filtered = filter_japanese_texts(mixed_texts)
    print(f"    入力: {mixed_texts}")
    print(f"    フィルタ後: {filtered}")

    # 3. 小規模テスト（APIを使用）
    if os.environ.get('GOOGLE_CLOUD_API_KEY'):
        print(f"\n  📋 generate_samples 小規模テスト:")
        test_dir = "./test_generate_samples"
        os.makedirs(test_dir, exist_ok=True)

        # 少数のサンプルでテスト
        test_texts = ["トモニ", "コモリ"]
        print(f"    テスト用テキスト: {test_texts}")
        print(f"    出力ディレクトリ: {test_dir}")

        try:
            result = generate_samples(
                text=test_texts,
                max_samples=3,  # 少数でテスト
                output_dir=test_dir
            )

            # 結果確認
            wav_files = list(Path(test_dir).glob("*.wav"))
            print(f"    結果: {result} samples generated")
            print(f"    WAVファイル数: {len(wav_files)}")

            if len(wav_files) > 0:
                print("    ✅ generate_samples テスト成功！")
            else:
                print("    ❌ WAVファイルが生成されませんでした")

        except Exception as e:
            print(f"    ❌ generate_samples テストでエラー: {e}")
            import traceback
            print(f"    詳細エラー: {traceback.format_exc()}")
    else:
        print(f"\n  ⚠️ APIキーが設定されていないため、generate_samplesのテストをスキップします")

except ImportError as e:
    print(f"  ❌ generate_samples のインポートに失敗: {e}")
    print(f"  現在のディレクトリ: {os.getcwd()}")
    print(f"  piper-sample-generator ディレクトリ存在: {os.path.exists('piper-sample-generator')}")
    if os.path.exists('piper-sample-generator'):
        print(f"  ディレクトリ内容: {os.listdir('piper-sample-generator')}")
except Exception as e:
    print(f"  ❌ テスト実行中にエラー: {e}")
    import traceback
    print(f"  詳細エラー: {traceback.format_exc()}")

# 最終確認
print(f"\n📋 最終システム状態:")
print(f"  日本語モジュール利用可能: {JAPANESE_MODULE_AVAILABLE}")
print(f"  data_ja.py 存在: {os.path.exists(data_ja_path)}")
print(f"  train.py 存在: {os.path.exists(train_py_path)}")
print(f"  generate_samples.py 存在: {os.path.exists('./piper-sample-generator/generate_samples.py')}")

if JAPANESE_MODULE_AVAILABLE:
    print(f"\n✅ すべてのコンポーネントが正常に動作しています！")
    print(f"   次のセルで日本語ウェイクワードのテスト音声生成を試してください。")
else:
    print(f"\n⚠️ 日本語モジュールのインポートに問題があります。")
    print(f"   Cell 4を再実行してみてください。")

🔍 システム診断:
  現在のディレクトリ: /content
  Python実行パス: /usr/bin/python3
  sys.path (最初の5項目):
    0: /content/openwakeword
    1: /content/piper-sample-generator
    2: /content
    3: /env/python
    4: /usr/lib/python311.zip

📁 ディレクトリ構造確認:
  ./openwakeword 存在: True
  ./openwakeword/openwakeword 存在: True
  ./openwakeword/openwakeword/data_ja.py 存在: True
  data_ja.py サイズ: 15211 bytes
🔄 既存のモジュールキャッシュをクリア

🔍 data_ja モジュールインポートテスト:
  Step 1: openwakeword パッケージをインポート
  ✅ openwakeword package path: /content/openwakeword/openwakeword/__init__.py
  Step 2: openwakeword.data_ja をインポート
  ✅ openwakeword.data_ja のインポート成功
  Step 3: 機能テスト
  ✅ is_japanese('トモリ') = True
  Step 4: adversarial texts生成テスト
  ✅ 生成されたadversarial texts: 1
  例: ['トノリ']
  ✅ 日本語モジュール利用可能

🔧 train.py を修正中...
  日本語サポートを追加中...
  ✅ train.py を修正しました - 日本語adversarial text生成が有効になりました

🔍 generate_samples 関数のテストを実行中...
  piper-sample-generator path: /content/piper-sample-generator
  存在確認: True
  ✅ generate_samples モジュールのインポート成功

  📋 日本語判定テスト:
  

Generating Japanese samples: 100%|██████████| 3/3 [00:01<00:00,  2.93it/s]

    結果: 3 samples generated
    WAVファイル数: 3
    ✅ generate_samples テスト成功！

📋 最終システム状態:
  日本語モジュール利用可能: True
  data_ja.py 存在: True
  train.py 存在: True
  generate_samples.py 存在: True

✅ すべてのコンポーネントが正常に動作しています！
   次のセルで日本語ウェイクワードのテスト音声生成を試してください。


In [6]:
# @title 6. 日本語ウェイクワードのテスト音声生成 { display-mode: "form" }
# @markdown 日本語のウェイクワードを入力して、正しく発音されるか確認しましょう。
# @markdown
# @markdown 発音のヒント:
# @markdown - ひらがな、カタカナ、漢字が使用できます
# @markdown - 2〜4音節が最適です（例：「トモリ」「アレクサ」「オーケーグーグル」）
# @markdown - 日常会話でよく使われる言葉は避けましょう

import sys
from IPython.display import Audio
import numpy as np
import scipy.io.wavfile
import time

target_word = 'トモリ' # @param {type:"string"}

def generate_japanese_test_audio(text):
    """日本語のテスト音声を生成"""
    if not os.environ.get('GOOGLE_CLOUD_API_KEY'):
        print("❌ APIキーが設定されていません")
        return None

    try:
        from google.cloud import texttospeech
    except ImportError:
        print("⚠️ google-cloud-texttospeechがインストールされていません。インストール中...")
        !pip install google-cloud-texttospeech -q
        from google.cloud import texttospeech

    client = texttospeech.TextToSpeechClient(
        client_options={"api_key": os.environ['GOOGLE_CLOUD_API_KEY']}
    )

    # 日本語の音声を使用（Chirp3 HD高品質音声）
    synthesis_input = texttospeech.SynthesisInput(text=text)
    voice = texttospeech.VoiceSelectionParams(
        language_code="ja-JP",
        name="ja-JP-Chirp3-HD-Aoede"  # 女性の高品質音声
    )
    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.LINEAR16,
        sample_rate_hertz=16000
    )

    try:
        response = client.synthesize_speech(
            input=synthesis_input,
            voice=voice,
            audio_config=audio_config
        )

        # 音声保存
        audio_array = np.frombuffer(response.audio_content, dtype=np.int16)
        scipy.io.wavfile.write("test_generation.wav", 16000, audio_array)
        print(f"✅ '{text}' の音声を生成しました（Chirp3 HD高品質音声）")
        return True
    except Exception as e:
        print(f"❌ エラー: {e}")
        # Chirp3 HDが使えない場合は標準音声にフォールバック
        try:
            voice = texttospeech.VoiceSelectionParams(
                language_code="ja-JP",
                name="ja-JP-Standard-A"
            )
            response = client.synthesize_speech(
                input=synthesis_input,
                voice=voice,
                audio_config=audio_config
            )
            audio_array = np.frombuffer(response.audio_content, dtype=np.int16)
            scipy.io.wavfile.write("test_generation.wav", 16000, audio_array)
            print(f"✅ '{text}' の音声を生成しました（標準音声）")
            return True
        except Exception as e2:
            print(f"❌ 標準音声でもエラー: {e2}")
            return False

if generate_japanese_test_audio(target_word):
    Audio("test_generation.wav", autoplay=True)
else:
    print("\n⚠️ 音声生成に失敗しました。APIキーを確認してください。")

✅ 'トモリ' の音声を生成しました（Chirp3 HD高品質音声）


In [ ]:
# @title 7. モデルの学習 (デバッグ強化版) { display-mode: "form" }
# @markdown # 7. モデルの学習
# @markdown ウェイクワードの確認とデータのダウンロードが完了しました。
# @markdown 最後に学習パラメータを調整（またはデフォルト値を使用）して学習を開始します！

# @markdown パラメータの説明:
# @markdown - `学習サンプル数`: ウェイクワードの例をいくつ生成するか（デフォルト1,000、最大50,000）
# @markdown - `学習ステップ数`: どのくらい長く学習するか（デフォルト10,000、最大50,000）
# @markdown - `誤検出ペナルティ`: 誤った検出をどの程度厳しく制限するか（高いほど誤検出が減りますが、認識率も下がる可能性があります）

# @markdown デフォルト値では、通常のCPUランタイムで約30〜60分かかります。
# @markdown より多くのサンプルや長い学習時間を使いたい場合は、GPUランタイムに変更してください。

# @markdown 学習が完了すると、`my_custom_model`フォルダに`.onnx`と`.tflite`ファイルが作成され、
# @markdown 自動的にダウンロードされます！

# デフォルトの設定ファイルを読み込み
import yaml
import os
import sys
from pathlib import Path

# 現在の作業ディレクトリを保存
work_dir = os.getcwd()
print(f"現在のディレクトリ: {work_dir}")

# openWakeWordディレクトリに移動
os.chdir("openwakeword")
oww_dir = os.getcwd()
print(f"openWakeWordディレクトリ: {oww_dir}")

# 設定ファイルを読み込み
config = yaml.load(open("examples/custom_model.yml", 'r').read(), yaml.Loader)

# パラメータ設定
学習サンプル数 = 1500 # @param {type:"slider", min:100, max:50000, step:50}
学習ステップ数 = 10000  # @param {type:"slider", min:0, max:50000, step:100}
誤検出ペナルティ = 1500  # @param {type:"slider", min:100, max:5000, step:50}

# 設定を更新（openWakeWordディレクトリからの相対パス）
config["target_phrase"] = [target_word]
config["model_name"] = target_word.replace(" ", "_")
config["n_samples"] = 学習サンプル数
config["n_samples_val"] = max(500, 学習サンプル数//10)
config["steps"] = 学習ステップ数
config["target_accuracy"] = 0.5
config["target_recall"] = 0.25
config["output_dir"] = "./my_custom_model"  # openWakeWordディレクトリからの相対パス
config["max_negative_weight"] = 誤検出ペナルティ

# パスの設定（すべてopenWakeWordディレクトリからの相対パス）
config["piper_sample_generator_path"] = "../piper-sample-generator"
config["background_paths"] = ['../audioset_16k', '../fma']
config["rir_paths"] = ["../mit_rirs"]
config["false_positive_validation_data_path"] = "../validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "../openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

# 設定ファイルを保存（openWakeWordディレクトリ内）
with open('training_config.yaml', 'w') as file:
    yaml.dump(config, file)

print(f"\n🎯 学習設定:")
print(f"  ウェイクワード: {target_word}")
print(f"  サンプル数: {学習サンプル数}")
print(f"  ステップ数: {学習ステップ数}")
print(f"  誤検出ペナルティ: {誤検出ペナルティ}")

# APIキーが設定されているか確認
if not os.environ.get('GOOGLE_CLOUD_API_KEY'):
    print("\n❌ エラー: Google Cloud APIキーが設定されていません！")
    print("最初のセルに戻ってAPIキーを設定してください。")
    # 作業ディレクトリに戻る
    os.chdir(work_dir)
else:
    print("\n🚀 学習を開始します...")
    print("=" * 60)

    # 既存のデータを確認
    model_dir = os.path.join("my_custom_model", config['model_name'])
    positive_train_dir = os.path.join(model_dir, "positive_train")
    positive_test_dir = os.path.join(model_dir, "positive_test")
    negative_train_dir = os.path.join(model_dir, "negative_train")
    negative_test_dir = os.path.join(model_dir, "negative_test")

    def check_directory_status(dir_path, name):
        """ディレクトリの状態をチェック"""
        if os.path.exists(dir_path):
            wav_files = list(Path(dir_path).glob("*.wav"))
            print(f"  {name}: {len(wav_files)} WAVファイル")
            return len(wav_files)
        else:
            print(f"  {name}: ディレクトリなし")
            return 0

    print(f"\n📋 既存データの確認:")
    check_directory_status(positive_train_dir, "positive_train")
    check_directory_status(positive_test_dir, "positive_test")
    check_directory_status(negative_train_dir, "negative_train")
    check_directory_status(negative_test_dir, "negative_test")

    # Step 1: 音声クリップを生成
    print(f"\n📣 Step 1/3: 音声サンプルを生成中...")
    print("=" * 40)

    try:
        !python openwakeword/train.py --training_config training_config.yaml --generate_clips
        print("✅ Step 1 完了")
    except Exception as e:
        print(f"❌ Step 1 でエラー: {e}")
        print("詳細なエラー情報を確認してください")

    # データ生成後の状態確認
    print(f"\n📋 Step 1 後のデータ確認:")
    pos_train_count = check_directory_status(positive_train_dir, "positive_train")
    pos_test_count = check_directory_status(positive_test_dir, "positive_test")
    neg_train_count = check_directory_status(negative_train_dir, "negative_train")
    neg_test_count = check_directory_status(negative_test_dir, "negative_test")

    # Step 2: クリップを拡張
    print(f"\n🎵 Step 2/3: 音声を拡張中...")
    print("=" * 40)

    # 必要な音声ファイルが存在するかチェック
    if pos_train_count > 0 and pos_test_count > 0 and neg_train_count > 0 and neg_test_count > 0:
        try:
            !python openwakeword/train.py --training_config training_config.yaml --augment_clips
            print("✅ Step 2 完了")
        except Exception as e:
            print(f"❌ Step 2 でエラー: {e}")
    else:
        print("❌ Step 2 スキップ: 必要な音声ファイルが不足しています")
        print(f"必要な最小ファイル数:")
        print(f"  positive_train: {pos_train_count} (>0 が必要)")
        print(f"  positive_test: {pos_test_count} (>0 が必要)")
        print(f"  negative_train: {neg_train_count} (>0 が必要)")
        print(f"  negative_test: {neg_test_count} (>0 が必要)")

        # 不足している場合の対処
        if neg_train_count == 0 or neg_test_count == 0:
            print("\n🔧 Negative sample不足の対処:")
            print("手動でnegative phrasesを設定してリトライします...")

            # 手動でnegative phrasesを追加
            config["custom_negative_phrases"] = ["トモニ", "コモリ", "トマリ", "ドモリ", "トボリ"]

            # 設定ファイルを再保存
            with open('training_config.yaml', 'w') as file:
                yaml.dump(config, file)

            print("設定を更新しました。Step 1からリトライします...")

            try:
                !python openwakeword/train.py --training_config training_config.yaml --generate_clips
                print("✅ リトライ後のStep 1 完了")

                # 再度状態確認
                print(f"\n📋 リトライ後のデータ確認:")
                neg_train_count = check_directory_status(negative_train_dir, "negative_train")
                neg_test_count = check_directory_status(negative_test_dir, "negative_test")

                if neg_train_count > 0 and neg_test_count > 0:
                    !python openwakeword/train.py --training_config training_config.yaml --augment_clips
                    print("✅ リトライ後のStep 2 完了")
                else:
                    print("❌ リトライ後もnegative samplesが生成されませんでした")

            except Exception as e:
                print(f"❌ リトライでもエラー: {e}")

    # 特徴量ファイルの確認
    print(f"\n📋 特徴量ファイルの確認:")
    feature_files = [
        "positive_features_train.npy",
        "positive_features_test.npy",
        "negative_features_train.npy",
        "negative_features_test.npy"
    ]

    missing_features = []
    for feature_file in feature_files:
        feature_path = os.path.join(model_dir, feature_file)
        if os.path.exists(feature_path):
            print(f"  ✅ {feature_file}")
        else:
            print(f"  ❌ {feature_file}")
            missing_features.append(feature_file)

    # Step 3: モデルを学習
    print(f"\n🧠 Step 3/3: モデルを学習中...")
    print("=" * 40)

    if len(missing_features) == 0:
        try:
            !python openwakeword/train.py --training_config training_config.yaml --train_model
            print("✅ Step 3 完了")

            # ONNXモデルをtfliteに変換
            print("\n🔄 tflite形式に変換中...")
            onnx_model_path = f"my_custom_model/{config['model_name']}.onnx"
            if os.path.exists(onnx_model_path):
                name1, name2 = f"my_custom_model/{config['model_name']}_float32.tflite", f"my_custom_model/{config['model_name']}.tflite"
                !onnx2tf -i {onnx_model_path} -o my_custom_model/ -kat onnx____Flatten_0 -ow -osd -f32 -nuo
                !mv {name1} {name2} 2>/dev/null || true

                print("\n✅ 学習が完了しました！")

                # 作業ディレクトリに戻る
                os.chdir(work_dir)

                # モデルファイルを自動ダウンロード
                try:
                    from google.colab import files
                    print("\n📥 モデルファイルをダウンロード中...")
                    model_onnx_path = f"openwakeword/my_custom_model/{config['model_name']}.onnx"
                    model_tflite_path = f"openwakeword/my_custom_model/{config['model_name']}.tflite"

                    if os.path.exists(model_onnx_path):
                        files.download(model_onnx_path)
                    if os.path.exists(model_tflite_path):
                        files.download(model_tflite_path)
                    print("\n🎉 完了！モデルファイルがダウンロードされました。")
                except:
                    print("\n📁 モデルファイルは以下の場所に保存されています：")
                    print(f"  - openwakeword/my_custom_model/{config['model_name']}.onnx")
                    print(f"  - openwakeword/my_custom_model/{config['model_name']}.tflite")
            else:
                os.chdir(work_dir)
                print("\n❌ エラー: ONNXモデルファイルが見つかりません。")

        except Exception as e:
            print(f"❌ Step 3 でエラー: {e}")
            os.chdir(work_dir)
    else:
        print(f"❌ Step 3 スキップ: 以下の特徴量ファイルが不足しています:")
        for missing in missing_features:
            print(f"  - {missing}")

        print("\n🔧 修正方法:")
        print("1. 上記のStep 1, 2が正常に完了していることを確認")
        print("2. negative samplesが正しく生成されていることを確認")
        print("3. すべてのディレクトリにWAVファイルが存在することを確認")

        os.chdir(work_dir)

print(f"\n📋 最終確認 - 作業ディレクトリに戻りました: {os.getcwd()}")

現在のディレクトリ: /content
openWakeWordディレクトリ: /content/openwakeword

🎯 学習設定:
  ウェイクワード: トモリ
  サンプル数: 1500
  ステップ数: 10000
  誤検出ペナルティ: 1500

🚀 学習を開始します...

📋 既存データの確認:
  positive_train: ディレクトリなし
  positive_test: ディレクトリなし
  negative_train: ディレクトリなし
  negative_test: ディレクトリなし

📣 Step 1/3: 音声サンプルを生成中...
2025-07-29 11:28:21.588580: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-29 11:28:21.604858: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753788501.626548    3589 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753788501.63299